# TP3 : Implémentation d'un système RAG (Retrieval-Augmented Generation)

**Étudiante :** Jihane — **Document utilisé :** mon CV (`pdfs/cv.pdf`)

Pipeline réalisé : Loading → Splitting → Embedding → Vector store (Chroma) → Retriever → Prompt → LLM → Évaluation (LLM-juge, *groundedness*).

# 0. Préparation de l'environnement

Dans le terminal du projet (une seule fois) :

```
uv init
uv venv
.venv\Scripts\activate
uv add ipykernel langchain langchain-text-splitters langchain-community pypdf tiktoken chromadb langchain-chroma python-dotenv pandas
uv add langchain-google-genai     # option Gemini
uv add langchain-ollama           # option Ollama (modèles locaux)
```

Créer un fichier `.env` à la racine (⚠️ ne jamais le pousser sur GitHub) :

```
GOOGLE_API_KEY=ta_cle_ici
```

In [1]:
import os, re
from dotenv import load_dotenv
from IPython.display import Markdown, display

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, PyPDFDirectoryLoader
from langchain_chroma import Chroma

load_dotenv(override=True)
print("Clé Gemini trouvée :", bool(os.getenv("GOOGLE_API_KEY")))

C:\Users\jihan\AppData\Local\Temp\ipykernel_16780\271900710.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyPDFDirectoryLoader


Clé Gemini trouvée : True


## LLM, modèle d'embedding et juge

Je n'ai pas de clé OpenAI, donc je choisis **Gemini** (ou **Ollama** en local). Change juste `PROVIDER`.
Les noms de modèles changent souvent : si une erreur « model not found » apparaît, modifie les noms ci-dessous.

In [18]:
PROVIDER = "gemini"          # "gemini" ou "ollama"

if PROVIDER == "gemini":
    from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
    LLM_NAME   = "gemini-3.1-flash-lite"    # le LLM du RAG
    JUDGE_NAME = "gemini-3-flash-preview"   # le juge : plus puissant que le LLM du RAG
    EMB_NAME   = "gemini-embedding-001"
    llm = ChatGoogleGenerativeAI(model=LLM_NAME, temperature=0)
    groundness_checker = ChatGoogleGenerativeAI(model=JUDGE_NAME, temperature=0)
    embedding_model = GoogleGenerativeAIEmbeddings(model=EMB_NAME)
else:
    from langchain_ollama import ChatOllama, OllamaEmbeddings
    # avant : ollama pull llama3.2 ; ollama pull llama3.1:8b ; ollama pull nomic-embed-text
    llm = ChatOllama(model="llama3.2", temperature=0)
    groundness_checker = ChatOllama(model="llama3.1:8b", temperature=0)
    embedding_model = OllamaEmbeddings(model="nomic-embed-text")


def to_text(msg):
    """Extrait le texte d'une réponse du LLM (que ce soit une chaîne ou une liste de blocs)."""
    c = msg.content
    if isinstance(c, str):
        return c
    return "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in c)


print(to_text(llm.invoke("Dis bonjour en une phrase.")))

Bonjour, j'espère que vous passez une excellente journée !


# 1. Loading du PDF et Chunking

In [4]:
pdf_file = "./pdfs/cv.pdf"          # <- mon CV (à placer dans le dossier pdfs/)
pdf_loader = PyPDFLoader(pdf_file)
# Pour charger tout un dossier : PyPDFDirectoryLoader(path="./pdfs")
pages = pdf_loader.load()
print("Nombre de pages :", len(pages))

Nombre de pages : 1


Découpage avec `RecursiveCharacterTextSplitter` + encodeur `tiktoken` (`o200k_base`) :
chunks de **300 tokens**, chevauchement (*overlap*) de **20**. (Tiktoken ne sert ici qu'à *compter* les tokens, ça marche avec n'importe quel LLM.)

In [5]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="o200k_base", chunk_size=300, chunk_overlap=20
)
chunks = text_splitter.split_documents(pages)
print("Nombre de chunks :", len(chunks))
print("\n--- Exemple de chunk ---\n", chunks[0].page_content)

Nombre de chunks : 5

--- Exemple de chunk ---
 Jihane Salek  
Étudiante en  Master Intelligence Artificielle et Digital Computing
jihanesalek263@gmail.com
 
07 28 09 12 25
 
Béni Mellal
 
PROFIL
Étudiante en Master Intelligence Artificielle et Digital Computing, je mʼintéresse autant au Deep Learning (Transformers, XAI) quʼaux 
architectures Big Data temps réel (Kafka, Federated Learning). Mon objectif : concevoir des systèmes intelligents, interprétables et utiles, en 
combinant recherche appliquée et développement concret.
FORMATION
Master Intelligence Artificielle et Digital Computing, Faculté des Sciences et Techniques Béni Mellal 09/2025 – Présent  | Béni Mellal
Licence en Informatique, Faculté des Sciences et Techniques Béni Mellal 09/2024 – 07/2025  | Béni Mellal
DEUST MIPC (Mathématiques, Informatique, Physique, Chimie)
Faculté des Sciences et Techniques Béni Mellal
09/2021 – 07/2024  | Béni Mellal
Baccalauréat Sciences Mathématiques B – Option Français, Lycée Technique Mohamm

> Mon CV est court, donc il y aura peu de chunks. Pour l'analyse (document de réponses), je peux aussi tester `chunk_size=100` / `overlap=0` et comparer.

# 2. Vector store : ChromaDB et Embeddings

In [6]:
os.makedirs("./store", exist_ok=True)
COLLECTION = "cv_jihane"

# On repart d'une collection vide à chaque exécution (sinon les chunks se dupliquent)
try:
    Chroma(collection_name=COLLECTION, embedding_function=embedding_model,
           persist_directory="./store").delete_collection()
except Exception as e:
    print("Rien à supprimer :", e)

vectorstore = Chroma.from_documents(
    chunks, embedding_model, collection_name=COLLECTION, persist_directory="./store"
)
print("Chunks indexés :", vectorstore._collection.count())

Chunks indexés : 5


# 3. Retriever

⚠️ Dans le notebook du prof il y avait une petite erreur : `search_Kwargs={'k',5}` (mauvaise casse + **set** au lieu de **dict**).
Du coup `k` était ignoré et le retriever renvoyait **4** chunks (valeur par défaut) au lieu de 5. Version corrigée : `search_kwargs={"k": 5}`.

In [7]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

test_q = "Quelles sont mes compétences techniques ?"
retrieved_chunks = retriever.invoke(test_q)
print("Nombre de chunks récupérés :", len(retrieved_chunks))
for i, d in enumerate(retrieved_chunks, 1):
    print(f"\n[{i}] page {d.metadata.get('page_label')} :\n{d.page_content[:300]}")

Nombre de chunks récupérés : 5

[1] page 1 :
Optimisation des hyperparamètres
Langages: Python, Java, JavaScript, SQL, C++
Frameworks IA: PyTorch, PyTorch Lightning, Hugging Face Transformers, Scikit-learn, XGBoost, SHAP, LIME, Pandas, NumPy, OpenCV, NLTK,
Matplotlib
Big Data / Web: Apache Kafka, Apache Spark, Hadoop, TimescaleDB, React, FastA

[2] page 1 :
•Vectorisation TF-IDF, entraînement de 6 classifieurs (SVM, LR, RF, DT, NB, Ridge) et optimisation des hyperparamètres via Grid Search, Random 
Search, Bayesian Optimization, PSO et Algorithmes Génétiques.
•Meilleur résultat : SVM avec 94,99% dʼaccuracy (vs 78% dans lʼarticle de référence).
•Technol

[3] page 1 :
•Automatisation de lʼextraction des données bibliographiques Scopus et web scraping SJR via SeleniumBase pour enrichir les publications de 
leurs quartiles.
•Développement des modules de gestion des concours, évaluation des candidats et génération de rapports (PDF/Excel).
•Technologies : Python, Fla

[4] page 1 :
EXPÉRIENCE

# 4. Prompt Design

In [8]:
prompt_template = """
Answer the following question based only on the provided context.
The context is extracted from the CV (resume) of Jihane, a Master student.
The context is delimited by <context> tag.
The user question is delimited by <question> tag.
Answer in the same language as the question.
If the answer is not found in the context, answer exactly: Je ne sais pas !
<context>
{context}
</context>
<question>
{question}
</question>
"""

Le prompt contient : (1) le message système (consignes + règle anti-hallucination « Je ne sais pas ! »),
(2) le contexte entre `<context>`, (3) la question entre `<question>`.

In [9]:
user_input = "Quelles sont mes compétences techniques ?"

relevant_document_chunks = retriever.invoke(user_input)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)

prompt = prompt_template.format(context=context_for_query, question=user_input)
print(prompt)


Answer the following question based only on the provided context.
The context is extracted from the CV (resume) of Jihane, a Master student.
The context is delimited by <context> tag.
The user question is delimited by <question> tag.
Answer in the same language as the question.
If the answer is not found in the context, answer exactly: Je ne sais pas !
<context>
Optimisation des hyperparamètres
Langages: Python, Java, JavaScript, SQL, C++
Frameworks IA: PyTorch, PyTorch Lightning, Hugging Face Transformers, Scikit-learn, XGBoost, SHAP, LIME, Pandas, NumPy, OpenCV, NLTK,
Matplotlib
Big Data / Web: Apache Kafka, Apache Spark, Hadoop, TimescaleDB, React, FastAPI, Flask, Spring Boot, Streamlit, Gradio, REST APIs
Bases de données & Outils: MySQL, PostgreSQL, MongoDB, Git, GitHub, Docker, Google Colab, VS Code, IntelliJ IDEA, UML
LANGUES, QUALITÉS & ACTIVITÉS
Langues
Arabe (maternelle), Français (courant), Anglais (courant)
Qualités
Esprit dʼéquipe, Autonomie, Adaptabilité, Sens de lʼorgani

# 5. Réponse du LLM

In [11]:
resp = llm.invoke(prompt)
display(Markdown(to_text(resp)))

D'après le document fourni, vos compétences techniques sont les suivantes :

**Langages :** Python, Java, JavaScript, SQL, C++

**Frameworks IA :** PyTorch, PyTorch Lightning, Hugging Face Transformers, Scikit-learn, XGBoost, SHAP, LIME, Pandas, NumPy, OpenCV, NLTK, Matplotlib

**Big Data / Web :** Apache Kafka, Apache Spark, Hadoop, TimescaleDB, React, FastAPI, Flask, Spring Boot, Streamlit, Gradio, REST APIs

**Bases de données & Outils :** MySQL, PostgreSQL, MongoDB, Git, GitHub, Docker, Google Colab, VS Code, IntelliJ IDEA, UML

**Domaines d'expertise :**
*   **IA / Data Science :** Machine Learning, Deep Learning, NLP, Transformers, Computer Vision, Explicabilité (XAI), Apprentissage fédéré, Optimisation des hyperparamètres.
*   **Autres :** Développement logiciel, Web scraping, Automatisation.

# 6. Regroupement dans une fonction RAG

`RAG(query)` : retriever → concaténation du contexte → prompt → LLM → réponse.

In [12]:
def RAG(query, llm=llm, prompt_template=prompt_template):
    context_docs = retriever.invoke(query)
    context_list = [d.page_content for d in context_docs]
    context_for_query = ". ".join(context_list)
    prompt = prompt_template.format(context=context_for_query, question=query)
    resp = llm.invoke(prompt)
    return to_text(resp)

display(Markdown(RAG("Quel est mon parcours académique ?")))

Votre parcours académique est le suivant :

*   **Master Intelligence Artificielle et Digital Computing** : Faculté des Sciences et Techniques Béni Mellal (09/2025 – Présent).
*   **Licence en Informatique** : Faculté des Sciences et Techniques Béni Mellal (09/2024 – 07/2025).
*   **DEUST MIPC (Mathématiques, Informatique, Physique, Chimie)** : Faculté des Sciences et Techniques Béni Mellal (09/2021 – 07/2024).
*   **Baccalauréat Sciences Mathématiques B – Option Français** : Lycée Technique Mohammed V (09/2020 – 07/2021).

### Tests sur mon CV (dont une question hors contexte pour vérifier le « Je ne sais pas ! »)

In [13]:
questions = [
    "Quelles sont mes compétences techniques ?",
    "Quel est mon parcours académique ?",
    "Quels projets ai-je réalisés ?",
    "Quelle est la capitale de l'Australie ?",   # hors contexte -> doit répondre 'Je ne sais pas !'
]
for q in questions:
    print("="*80, "\nQ :", q)
    display(Markdown(RAG(q)))

Q : Quelles sont mes compétences techniques ?


D'après le document fourni, vos compétences techniques sont les suivantes :

**Langages :** Python, Java, JavaScript, SQL, C++

**Frameworks IA :** PyTorch, PyTorch Lightning, Hugging Face Transformers, Scikit-learn, XGBoost, SHAP, LIME, Pandas, NumPy, OpenCV, NLTK, Matplotlib

**Big Data / Web :** Apache Kafka, Apache Spark, Hadoop, TimescaleDB, React, FastAPI, Flask, Spring Boot, Streamlit, Gradio, REST APIs

**Bases de données & Outils :** MySQL, PostgreSQL, MongoDB, Git, GitHub, Docker, Google Colab, VS Code, IntelliJ IDEA, UML

**Domaines d'expertise :**
*   **IA / Data Science :** Machine Learning, Deep Learning, NLP, Transformers, Computer Vision, Explicabilité (XAI), Apprentissage fédéré, Optimisation des hyperparamètres.
*   **Autres :** Développement logiciel, Web scraping, Automatisation.

Q : Quel est mon parcours académique ?


Votre parcours académique est le suivant :

*   **Master Intelligence Artificielle et Digital Computing** : Faculté des Sciences et Techniques Béni Mellal (09/2025 – Présent).
*   **Licence en Informatique** : Faculté des Sciences et Techniques Béni Mellal (09/2024 – 07/2025).
*   **DEUST MIPC (Mathématiques, Informatique, Physique, Chimie)** : Faculté des Sciences et Techniques Béni Mellal (09/2021 – 07/2024).
*   **Baccalauréat Sciences Mathématiques B – Option Français** : Lycée Technique Mohammed V (09/2020 – 07/2021).

Q : Quels projets ai-je réalisés ?


Les projets que vous avez réalisés sont les suivants :

*   **Plateforme Scientific Ranking** (Projet de Fin d'Études) : Conception d'une plateforme web automatisant l'évaluation et le classement des dossiers de chercheurs pour les concours universitaires.
*   **Système intelligent de prédiction énergétique hospitalière en temps réel** : Pipeline Big Data avec Apache Kafka, prédiction multi-horizon avec XGBoost et apprentissage fédéré.
*   **Analyse des sentiments en langue arabe** : Conception d'un pipeline NLP complet et entraînement de 6 classifieurs (meilleur résultat avec SVM).
*   **Détection multilingue de Fake News** : Fine-tuning du modèle XLM-RoBERTa avec PyTorch Lightning et intégration de méthodes d'explicabilité (XAI).
*   **Projet Offre Detector** (réalisé lors du stage chez DBM MAROC) : Développement d'un module Produits & Services et conception d'un système d'estimation intelligente des prix basé sur des recherches Web multi-sources.

Q : Quelle est la capitale de l'Australie ?


Je ne sais pas !

# 7. Évaluation de la réponse du LLM (LLM-as-a-judge)

![architecture](./rag_juge.gif)

Un second LLM (plus puissant) joue le **juge** : on lui donne la question, le contexte et la réponse, et il note la *groundedness*
(la réponse est-elle dérivée **uniquement** du contexte ?) de 1 à 5.

In [14]:
user_message_template = """
###Question
{question}
###Context
{context}
###Answer
{answer}
"""

## 7.2 Groundedness

In [15]:
groundness_rater_system_message = """
Vous êtes chargé d'évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d'IA pour générer la réponse, ainsi qu'une réponse générée par l'IA à la question.
Dans l'entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par l'IA commencera par ###Answer.
Critères d'évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1- La métrique n'est pas respectée du tout
2- La métrique n'est respectée que dans une mesure limitée
3- La métrique est respectée dans une bonne mesure
4- La métrique est respectée en grande partie
5- La métrique est entièrement respectée

Métrique :

La réponse doit être dérivée uniquement des informations présentées dans le contexte.

Instructions:

Écrivez d'abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d'évaluation et attribuer un score.
Terminez TOUJOURS par une dernière ligne au format exact : Score: X   (X entier de 1 à 5)
"""

In [19]:
def evaluate(system_message, user_message_template, question, model=groundness_checker):
    retrieved_chunks = retriever.invoke(question)
    context = ". ".join(d.page_content for d in retrieved_chunks)
    answer = RAG(question)
    prompt = f"""
{system_message}

USER:
{user_message_template.format(question=question, context=context, answer=answer)}
"""
    juge_response = model.invoke(prompt)
    return answer, to_text(juge_response)

def extract_score(text):
    m = re.findall(r"Score\s*:\s*\**\s*([1-5])", text)
    return int(m[-1]) if m else None

answer, verdict = evaluate(groundness_rater_system_message, user_message_template, questions[0])
print("Réponse du RAG :", answer)
display(Markdown(verdict))
print("Score :", extract_score(verdict))

Réponse du RAG : D'après le document fourni, vos compétences techniques sont les suivantes :

**Langages :** Python, Java, JavaScript, SQL, C++

**Frameworks IA :** PyTorch, PyTorch Lightning, Hugging Face Transformers, Scikit-learn, XGBoost, SHAP, LIME, Pandas, NumPy, OpenCV, NLTK, Matplotlib

**Big Data / Web :** Apache Kafka, Apache Spark, Hadoop, TimescaleDB, React, FastAPI, Flask, Spring Boot, Streamlit, Gradio, REST APIs

**Bases de données & Outils :** MySQL, PostgreSQL, MongoDB, Git, GitHub, Docker, Google Colab, VS Code, IntelliJ IDEA, UML

**Domaines d'expertise :**
*   **IA / Data Science :** Machine Learning, Deep Learning, NLP, Transformers, Computer Vision, Explicabilité (XAI), Apprentissage fédéré, Optimisation des hyperparamètres.
*   **Autres :** Développement logiciel, Web scraping, Automatisation.


Pour évaluer la réponse générée par l'IA, nous allons suivre les étapes suivantes :

1.  **Identifier les compétences techniques mentionnées dans le contexte :** Parcourir les sections "Langages", "Frameworks IA", "Big Data / Web", "Bases de données & Outils", "COMPÉTENCES" et les descriptions de projets/expériences pour lister les technologies et domaines cités.
2.  **Vérifier chaque élément de la réponse par rapport au contexte :** S'assurer que chaque langage, framework, outil ou domaine d'expertise listé dans la réponse figure explicitement dans le texte source.
3.  **Rechercher des informations extrinsèques :** Vérifier si l'IA a ajouté des compétences qui ne sont pas présentes dans le document (hallucinations ou connaissances générales non mentionnées).
4.  **Attribuer un score :** En fonction de la fidélité de la réponse au contexte.

---

**Explication étape par étape :**

*   **Langages :** La réponse liste Python, Java, JavaScript, SQL, C++. Tous ces langages sont explicitement listés dans la section "Langages" du contexte.
*   **Frameworks IA :** La réponse liste PyTorch, PyTorch Lightning, Hugging Face Transformers, Scikit-learn, XGBoost, SHAP, LIME, Pandas, NumPy, OpenCV, NLTK, Matplotlib. Tous ces éléments sont présents dans la section "Frameworks IA" du contexte.
*   **Big Data / Web :** La réponse cite Apache Kafka, Apache Spark, Hadoop, TimescaleDB, React, FastAPI, Flask, Spring Boot, Streamlit, Gradio, REST APIs. Tous ces éléments figurent dans la section "Big Data / Web" du contexte.
*   **Bases de données & Outils :** La réponse mentionne MySQL, PostgreSQL, MongoDB, Git, GitHub, Docker, Google Colab, VS Code, IntelliJ IDEA, UML. Tous sont présents dans la section correspondante du contexte.
*   **Domaines d'expertise (IA / Data Science) :** La réponse cite Machine Learning, Deep Learning, NLP, Transformers, Computer Vision, Explicabilité (XAI), Apprentissage fédéré, Optimisation des hyperparamètres. Ces termes se retrouvent dans les sections "COMPÉTENCES", "PROFIL" et dans le détail des projets (ex: "Optimisation des hyperparamètres" est le premier élément du contexte).
*   **Autres :**
    *   *Développement logiciel* : Mentionné dans l'expérience chez DBM MAROC ("Développement logiciel & Intelligence Artificielle").
    *   *Web scraping* : Mentionné dans la partie sur SeleniumBase ("web scraping SJR via SeleniumBase").
    *   *Automatisation* : Mentionné dans la partie sur SeleniumBase ("Automatisation de l'extraction").

La réponse est une synthèse fidèle et structurée des informations textuelles fournies. Aucune information extérieure au contexte n'a été ajoutée.

---

**Évaluation de la métrique :**

La métrique est entièrement respectée. Chaque élément de la réponse trouve sa source directe dans le texte fourni.

Score: 5

Score : 5


### Évaluation de toutes les questions (tableau pour le document de réponses)

In [20]:
import pandas as pd

rows = []
for q in questions:
    ans, verdict = evaluate(groundness_rater_system_message, user_message_template, q)
    rows.append({"question": q, "réponse RAG": ans[:150].replace("\n", " "), "score": extract_score(verdict)})

df = pd.DataFrame(rows)
df

,question,réponse RAG,score
0,Quelles sont mes compétences techniques ?,"D'après le document fourni, vos compétences te...",5
1,Quel est mon parcours académique ?,Votre parcours académique est le suivant : * ...,5
2,Quels projets ai-je réalisés ?,Les projets que vous avez réalisés sont les su...,5
3,Quelle est la capitale de l'Australie ?,Je ne sais pas !,5


### Échelle de notation (fidélité au contexte)

| Score | Signification | Description |
| :--- | :--- | :--- |
| **5 / 5** | Excellent | Uniquement les informations du contexte. Aucune hallucination. |
| **4 / 5** | Bon | Correct et appuyé sur le contexte, formulation un peu moins précise. |
| **3 / 5** | Moyen | Partiellement vrai, omet un détail important ou manque de clarté. |
| **2 / 5** | Médiocre | Extrapolations / informations non vérifiables avec le contexte. |
| **1 / 5** | Mauvais | Hors-sujet, faux ou inventé. |